# EMCAD ClinicDB — Independent Experimental Reproduction

This notebook documents an implementation-based reproduction of **PVT-EMCAD-B2** from:

> Rahman, M. M., Munir, M., & Marculescu, R.  
> *EMCAD: Efficient Multi-scale Convolutional Attention Decoding for Medical Image Segmentation.*  
> CVPR 2024.

**Objective:** reproduce the reported ClinicDB segmentation result using the authors' released EMCAD implementation, while documenting the computational environment, training procedure, checkpoint selection, standalone test evaluation, and qualitative results.

### Reproduction result

| Setting | Published | This reproduction |
|---|---:|---:|
| Model | PVT-EMCAD-B2 | PVT-EMCAD-B2 |
| Dataset | ClinicDB | ClinicDB |
| Input size | 352 × 352 | 352 × 352 |
| Epochs | 200 | 200 |
| Batch size | 16 | 4 |
| GPU | RTX A6000 48 GB | Kaggle Tesla T4 |
| Dice | **95.21%** (5-run average) | **95.02%** (1 run) |

The reduced batch size reflects the available GPU memory. Therefore, this is an **implementation-based reproduction under constrained computational conditions**, not an exact hardware-level reproduction.


## 1. Reproducibility Configuration

The paths below correspond to the Kaggle dataset used for this experiment. All generated files are written to `/kaggle/working`.


In [ ]:
from pathlib import Path
import os
import shutil
import sys

# ---------------------------------------------------------------------
# Kaggle input/output configuration
# ---------------------------------------------------------------------

INPUT_ROOT = Path(
    "/kaggle/input/datasets/audityghosh/emcad-polyp-pretrained/EMCAD-main"
)
WORK_ROOT = Path("/kaggle/working/EMCAD-main")

DATA_ROOT = INPUT_ROOT / "data" / "polyp" / "polyp"
CLINICDB_ROOT = DATA_ROOT / "ClinicDB"
TRAIN_ROOT = CLINICDB_ROOT / "train"
TEST_ROOT = CLINICDB_ROOT / "test"

PVT_ROOT = INPUT_ROOT / "pretrained_pth" / "pvt" / "pvt"
PVT_CHECKPOINT = PVT_ROOT / "pvt_v2_b2.pth"

CHECKPOINT_DIR = WORK_ROOT / "checkpoints"
PREDICTION_ROOT = WORK_ROOT / "predictions_final"
RESULT_ROOT = WORK_ROOT / "results_final"
QUALITATIVE_ROOT = WORK_ROOT / "qualitative_results"

IMAGE_SIZE = 352
BATCH_SIZE = 4
TEST_BATCH_SIZE = 4
NUM_EPOCHS = 200

print("Input root :", INPUT_ROOT)
print("Work root  :", WORK_ROOT)
print("ClinicDB   :", CLINICDB_ROOT)
print("PVT-B2     :", PVT_CHECKPOINT)


In [ ]:
# Validate the expected Kaggle dataset structure.

required_dirs = [
    INPUT_ROOT,
    DATA_ROOT,
    TRAIN_ROOT,
    TEST_ROOT,
    PVT_ROOT,
]

for path in required_dirs:
    assert path.is_dir(), f"Missing directory: {path}"

assert PVT_CHECKPOINT.is_file(), f"Missing checkpoint: {PVT_CHECKPOINT}"

print("Dataset and pretrained-weight structure verified.")
print(f"ClinicDB train: {TRAIN_ROOT}")
print(f"ClinicDB test : {TEST_ROOT}")
print(f"PVT-B2 weights: {PVT_CHECKPOINT}")


## 2. Environment and Dependencies

The original EMCAD implementation was developed with an older PyTorch environment. This reproduction was executed on Kaggle using the environment reported below.

The dependency installation is kept separate from the experiment itself so the notebook is easier to audit.


In [ ]:
# Install the Python dependencies used by the EMCAD implementation and evaluation code.
# PyTorch is intentionally not reinstalled here because Kaggle provides the
# CUDA-compatible PyTorch build used for the experiment.

%pip install -q \
    "numpy<2" \
    "albumentations==1.1.0" \
    "timm==0.6.13" \
    "segmentation-mask-overlay==0.3.4" \
    medpy \
    thop \
    ptflops \
    torchprofile \
    torchmetrics \
    torchsummary \
    torchsummaryx \
    warmup-scheduler \
    simpleitk \
    nibabel \
    ml_collections \
    tensorboardx \
    einops \
    loguru \
    tabulate

print("Dependencies installed.")


In [ ]:
import torch
import timm

print("PyTorch :", torch.__version__)
print("timm    :", timm.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("CUDA ver:", torch.version.cuda)


## 3. Prepare the EMCAD Source Tree

The official EMCAD source tree is copied into the writable Kaggle working directory. This keeps the original Kaggle input dataset untouched while allowing the training and evaluation scripts to be modified for the experiment.


In [ ]:
# Copy the released EMCAD repository into the writable working directory.

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

shutil.copytree(INPUT_ROOT, WORK_ROOT)

os.chdir(WORK_ROOT)

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

print("Working directory:", Path.cwd())
print("EMCAD source tree copied successfully.")


In [ ]:
from lib.networks import EMCADNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EMCADNet(
    num_classes=1,
    kernel_sizes=[1, 3, 5],
    expansion_factor=2,
    dw_parallel=True,
    add=True,
    lgag_ks=3,
    activation="relu6",
    encoder="pvt_v2_b2",
    pretrain=True,
    pretrained_dir=str(PVT_ROOT) + "/",
).to(device)

print("EMCADNet import and model construction successful.")
print("Device:", device)


## 4. Data Loader and Forward-Pass Smoke Test

Before starting the long training run, verify that:

1. ClinicDB images and masks can be loaded.
2. Images have the expected spatial resolution.
3. PVTv2-B2 pretrained weights are available.
4. EMCAD produces the expected multi-stage decoder outputs.


In [ ]:
from utils.dataloader_polyp import get_loader

TRAIN_IMAGES = TRAIN_ROOT / "images"
TRAIN_MASKS = TRAIN_ROOT / "masks"

train_loader = get_loader(
    image_root=str(TRAIN_IMAGES) + "/",
    gt_root=str(TRAIN_MASKS) + "/",
    batchsize=2,
    trainsize=IMAGE_SIZE,
    shuffle=True,
    augmentation=True,
    split="train",
    color_image=True,
)

images, masks = next(iter(train_loader))

print("Images:", images.shape, images.dtype)
print("Masks :", masks.shape, masks.dtype)

model.eval()
with torch.no_grad():
    outputs = model(images.to(device))

print("\nForward pass successful.")
for idx, output in enumerate(outputs, start=1):
    print(f"Decoder output {idx}: {tuple(output.shape)}")


## 5. Training Configuration

The released training script is adapted only where necessary for the Kaggle filesystem and checkpoint handling. The model configuration follows PVT-EMCAD-B2:

- Encoder: PVTv2-B2 with ImageNet-pretrained weights
- Decoder: EMCAD
- Multi-scale kernels: `[1, 3, 5]`
- Expansion factor: `2`
- Parallel depth-wise convolution: enabled
- LGAG kernel size: `3`
- Activation: `ReLU6`
- Input size: `352 × 352`
- Batch size: `4`
- Training length: `200` epochs
- Gradient clipping: `0.5`
- Optimizer: AdamW
- Weight decay: `1e-4`

The following cell writes the Kaggle-adapted training script used for the experiment.


In [ ]:
%%writefile /kaggle/working/EMCAD-main/train_polyp.py

import os
import sys
import time
import argparse
import glob

import torch
import torch.nn.functional as F
from torch.autograd import Variable
from torch.optim.lr_scheduler import CosineAnnealingLR

from lib.networks import EMCADNet
from utils.dataloader_polyp import get_loader
from utils.utils import clip_gradient, AvgMeter


# ============================================================
# KAGGLE PATHS
# ============================================================

ROOT = "/kaggle/working/EMCAD-main"

INPUT_ROOT = "/kaggle/input/datasets/audityghosh/emcad-polyp-pretrained/EMCAD-main"

DATA_ROOT = f"{INPUT_ROOT}/data/polyp/polyp"

TRAIN_PATH = f"{DATA_ROOT}/ClinicDB/train/"
TEST_PATH = f"{DATA_ROOT}/ClinicDB/"

PRETRAINED_DIR = (
    f"{INPUT_ROOT}/pretrained_pth/pvt/pvt/"
)

# All outputs MUST go to /kaggle/working
CHECKPOINT_DIR = f"{ROOT}/checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)


# ============================================================
# LOSS
# ============================================================

def structure_loss(pred, mask, w=1):
    weit = 1 + 5 * torch.abs(
        F.avg_pool2d(
            mask,
            kernel_size=31,
            stride=1,
            padding=15
        ) - mask
    )

    wbce = F.binary_cross_entropy_with_logits(
        pred,
        mask,
        reduction="none"
    )

    wbce = (
        (weit * wbce).sum(dim=(2, 3))
        / weit.sum(dim=(2, 3))
    )

    pred = torch.sigmoid(pred)

    inter = ((pred * mask) * weit).sum(dim=(2, 3))

    union = ((pred + mask) * weit).sum(dim=(2, 3))

    wiou = 1 - (
        (inter + 1)
        / (union - inter + 1)
    )

    return (w * (wbce + wiou)).mean()


# ============================================================
# DICE / IOU
# ============================================================

def dice_coefficient(predicted, labels):

    predicted = predicted.float()
    labels = labels.float().to(predicted.device)

    smooth = 1e-6

    predicted = predicted.contiguous().view(-1)
    labels = labels.contiguous().view(-1)

    intersection = (predicted * labels).sum()

    return (
        (2 * intersection + smooth)
        / (predicted.sum() + labels.sum() + smooth)
    )


def iou(predicted, labels):

    predicted = predicted.float()
    labels = labels.float().to(predicted.device)

    smooth = 1e-6

    predicted = predicted.contiguous().view(-1)
    labels = labels.contiguous().view(-1)

    intersection = (predicted * labels).sum()

    union = (
        predicted.sum()
        + labels.sum()
        - intersection
    )

    return (intersection + smooth) / (union + smooth)


# ============================================================
# EVALUATION
# ============================================================

def evaluate(model, path, dataset, opt):

    data_path = os.path.join(path, dataset)

    image_root = os.path.join(data_path, "images")
    gt_root = os.path.join(data_path, "masks")

    loader = get_loader(
        image_root=image_root,
        gt_root=gt_root,
        batchsize=opt.test_batchsize,
        trainsize=opt.img_size,
        shuffle=False,
        num_workers=opt.num_workers,
        pin_memory=True,
        augmentation=False,
        split="test",
        color_image=True
    )

    model.eval()

    total_dice = 0
    total_iou = 0
    total_images = 0

    with torch.no_grad():

        for images, gts, original_shapes, _ in loader:

            images = images.cuda(non_blocking=True)

            gts = gts.float().cuda(non_blocking=True)

            outputs = model(images)

            pred = outputs[-1]

            for i in range(images.size(0)):

                w_orig = int(original_shapes[0][i])
                h_orig = int(original_shapes[1][i])

                p = pred[i].unsqueeze(0)

                p = F.interpolate(
                    p,
                    size=(h_orig, w_orig),
                    mode="bilinear",
                    align_corners=False
                )

                p = torch.sigmoid(p).squeeze()

                pmin = p.min()
                pmax = p.max()

                p = (
                    (p - pmin)
                    / (pmax - pmin + 1e-8)
                )

                g = gts[i].unsqueeze(0)

                g = F.interpolate(
                    g,
                    size=(h_orig, w_orig),
                    mode="nearest"
                ).squeeze()

                pred_bin = (p >= 0.5).float()

                gt_bin = (g >= 0.2).float()

                total_dice += dice_coefficient(
                    pred_bin,
                    gt_bin
                ).item()

                total_iou += iou(
                    pred_bin,
                    gt_bin
                ).item()

                total_images += 1

    if total_images == 0:
        raise RuntimeError(
            f"No images found in {image_root}"
        )

    return (
        total_dice / total_images,
        total_iou / total_images
    )


# ============================================================
# TRAIN
# ============================================================

def train_one_epoch(
    model,
    train_loader,
    optimizer,
    epoch,
    opt
):

    model.train()

    loss_meter = AvgMeter()

    total_step = len(train_loader)

    size_rates = [0.75, 1, 1.25]

    for step, (images, gts) in enumerate(
        train_loader,
        start=1
    ):

        for rate in size_rates:

            optimizer.zero_grad(
                set_to_none=True
            )

            images = images.cuda(
                non_blocking=True
            )

            gts = gts.float().cuda(
                non_blocking=True
            )

            if rate != 1:

                trainsize = int(
                    round(
                        opt.img_size
                        * rate
                        / 32
                    ) * 32
                )

                images = F.interpolate(
                    images,
                    size=(trainsize, trainsize),
                    mode="bilinear",
                    align_corners=True
                )

                gts = F.interpolate(
                    gts,
                    size=(trainsize, trainsize),
                    mode="nearest"
                )

            outputs = model(images)

            loss_p1 = structure_loss(
                outputs[0],
                gts
            )

            loss_p2 = structure_loss(
                outputs[1],
                gts
            )

            loss_p3 = structure_loss(
                outputs[2],
                gts
            )

            loss_p4 = structure_loss(
                outputs[3],
                gts
            )

            loss_all = structure_loss(
                outputs[0]
                + outputs[1]
                + outputs[2]
                + outputs[3],
                gts
            )

            loss = (
                loss_p1
                + loss_p2
                + loss_p3
                + loss_p4
                + loss_all
            )

            loss.backward()

            clip_gradient(
                optimizer,
                opt.clip
            )

            optimizer.step()

            if rate == 1:

                loss_meter.update(
                    loss.detach(),
                    images.size(0)
                )

        if (
            step % 100 == 0
            or step == total_step
        ):

            print(
                f"Epoch [{epoch}/{opt.epoch}] "
                f"Step [{step}/{total_step}] "
                f"Loss: {loss_meter.show():.4f} "
                f"LR: "
                f"{optimizer.param_groups[0]['lr']:.8f}"
            )

    return loss_meter.show()


# ============================================================
# CHECKPOINT SAVE
# ============================================================

def save_checkpoint(
    model,
    optimizer,
    scheduler,
    epoch,
    best_val_dice,
    test_dice_at_best
):

    checkpoint = {

        "epoch": epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_val_dice":
            best_val_dice,

        "test_dice_at_best":
            test_dice_at_best
    }

    # ALWAYS save latest checkpoint
    last_path = os.path.join(
        CHECKPOINT_DIR,
        "EMCAD_ClinicDB_last_full.pth"
    )

    torch.save(
        checkpoint,
        last_path
    )

    return last_path


def save_best_checkpoint(
    model,
    optimizer,
    scheduler,
    epoch,
    best_val_dice,
    test_dice_at_best
):

    checkpoint = {

        "epoch": epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_val_dice":
            best_val_dice,

        "test_dice_at_best":
            test_dice_at_best
    }

    best_path = os.path.join(
        CHECKPOINT_DIR,
        "EMCAD_ClinicDB_best_full.pth"
    )

    torch.save(
        checkpoint,
        best_path
    )

    return best_path


# ============================================================
# FIND RESUME CHECKPOINT
# ============================================================

def find_checkpoint():

    candidates = []

    for root, dirs, files in os.walk(
        "/kaggle/input"
    ):

        for file in files:

            if file.endswith(
                (".pth", ".pt", ".ckpt")
            ):

                path = os.path.join(
                    root,
                    file
                )

                name = file.lower()

                if (
                    "best" in name
                    or "last" in name
                    or "checkpoint" in name
                    or "resume" in name
                ):

                    candidates.append(path)

    if not candidates:
        return None

    # Prefer FULL checkpoint names
    candidates.sort(
        key=lambda x: (
            "last_full" not in x.lower(),
            "best_full" not in x.lower(),
            x
        )
    )

    print("\nResume candidates:")

    for c in candidates:
        print(" ", c)

    return candidates[0]


# ============================================================
# LOAD CHECKPOINT
# ============================================================

def load_checkpoint(
    model,
    optimizer,
    scheduler,
    checkpoint_path
):

    print("\nLoading checkpoint:")
    print(checkpoint_path)

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cuda"
    )

    # Full checkpoint
    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        scheduler.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )

        start_epoch = int(
            checkpoint["epoch"]
        )

        best_val_dice = float(
            checkpoint.get(
                "best_val_dice",
                0
            )
        )

        test_dice_at_best = float(
            checkpoint.get(
                "test_dice_at_best",
                0
            )
        )

        print(
            f"FULL checkpoint loaded."
        )

        print(
            f"Previous epoch: {start_epoch}"
        )

        print(
            f"Best Val Dice: {best_val_dice:.4f}"
        )

        return (
            start_epoch,
            best_val_dice,
            test_dice_at_best
        )

    # Plain model weights
    else:

        model.load_state_dict(
            checkpoint
        )

        print(
            "Plain model weights loaded."
        )

        print(
            "Optimizer and epoch cannot "
            "be restored."
        )

        return 0, 0, 0


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--epoch",
        type=int,
        default=200
    )

    parser.add_argument(
        "--batchsize",
        type=int,
        default=4
    )

    parser.add_argument(
        "--test_batchsize",
        type=int,
        default=4
    )

    parser.add_argument(
        "--img_size",
        type=int,
        default=352
    )

    parser.add_argument(
        "--lr",
        type=float,
        default=0.0005
    )

    parser.add_argument(
        "--clip",
        type=float,
        default=0.5
    )

    parser.add_argument(
        "--num_workers",
        type=int,
        default=2
    )

    parser.add_argument(
        "--resume",
        action="store_true"
    )

    opt = parser.parse_args()

    print("=" * 70)
    print("EMCAD KAGGLE RESUMABLE TRAINING")
    print("=" * 70)

    print(
        "Train:",
        TRAIN_PATH
    )

    print(
        "Test:",
        TEST_PATH
    )

    print(
        "Pretrained:",
        PRETRAINED_DIR
    )

    # --------------------------------------------------------
    # CHECK PATHS
    # --------------------------------------------------------

    assert os.path.isdir(
        TRAIN_PATH
    )

    assert os.path.isdir(
        TEST_PATH
    )

    assert os.path.isfile(
        os.path.join(
            PRETRAINED_DIR,
            "pvt_v2_b2.pth"
        )
    )

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model = EMCADNet(
        num_classes=1,
        kernel_sizes=[1, 3, 5],
        expansion_factor=2,
        dw_parallel=True,
        add=True,
        lgag_ks=3,
        activation="relu6",
        encoder="pvt_v2_b2",
        pretrain=True,
        pretrained_dir=PRETRAINED_DIR
    )

    model = model.cuda()

    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=opt.lr,
        weight_decay=1e-4
    )

    # --------------------------------------------------------
    # SCHEDULER
    # --------------------------------------------------------

    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=opt.epoch,
        eta_min=1e-6
    )

    start_epoch = 0

    best_val_dice = 0

    test_dice_at_best = 0

    # --------------------------------------------------------
    # RESUME
    # --------------------------------------------------------

    if opt.resume:

        checkpoint_path = find_checkpoint()

        if checkpoint_path is not None:

            (
                start_epoch,
                best_val_dice,
                test_dice_at_best
            ) = load_checkpoint(
                model,
                optimizer,
                scheduler,
                checkpoint_path
            )

        else:

            print(
                "\nNo checkpoint found."
            )

            print(
                "Starting from epoch 0."
            )

    # --------------------------------------------------------
    # DATALOADER
    # --------------------------------------------------------

    train_loader = get_loader(
        image_root=f"{TRAIN_PATH}/images/",
        gt_root=f"{TRAIN_PATH}/masks/",
        batchsize=opt.batchsize,
        trainsize=opt.img_size,
        shuffle=True,
        num_workers=opt.num_workers,
        pin_memory=True,
        augmentation=True,
        split="train",
        color_image=True
    )

    print(
        "\nTraining images:",
        len(train_loader.dataset)
    )

    print(
        "Batches:",
        len(train_loader)
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    for epoch in range(
        start_epoch + 1,
        opt.epoch + 1
    ):

        print("\n")
        print("=" * 70)
        print(
            f"STARTING EPOCH {epoch}/{opt.epoch}"
        )
        print("=" * 70)

        epoch_start = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            epoch,
            opt
        )

        scheduler.step()

        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        val_dice, val_iou = evaluate(
            model,
            TEST_PATH,
            "val",
            opt
        )

        # ----------------------------------------------------
        # TEST
        # ----------------------------------------------------

        test_dice, test_iou = evaluate(
            model,
            TEST_PATH,
            "test",
            opt
        )

        print("\n")
        print("=" * 70)
        print(
            f"EPOCH {epoch} RESULT"
        )
        print("=" * 70)

        print(
            f"Train Loss : {train_loss:.6f}"
        )

        print(
            f"Val Dice   : {val_dice:.6f}"
        )

        print(
            f"Val IoU    : {val_iou:.6f}"
        )

        print(
            f"Test Dice  : {test_dice:.6f}"
        )

        print(
            f"Test IoU   : {test_iou:.6f}"
        )

        print(
            f"Time       : "
            f"{time.time() - epoch_start:.1f}s"
        )

        # ----------------------------------------------------
        # BEST MODEL
        # ----------------------------------------------------

        if val_dice > best_val_dice:

            best_val_dice = val_dice

            test_dice_at_best = test_dice

            best_path = save_best_checkpoint(
                model,
                optimizer,
                scheduler,
                epoch,
                best_val_dice,
                test_dice_at_best
            )

            print(
                "\n*** NEW BEST MODEL ***"
            )

            print(
                "Saved:",
                best_path
            )

        # ----------------------------------------------------
        # ALWAYS SAVE LAST
        # ----------------------------------------------------

        last_path = save_checkpoint(
            model,
            optimizer,
            scheduler,
            epoch,
            best_val_dice,
            test_dice_at_best
        )

        print(
            "\nLatest checkpoint:"
        )

        print(last_path)

        print(
            f"\nBest Val Dice so far: "
            f"{best_val_dice:.6f}"
        )

    print("\n")
    print("=" * 70)
    print("TRAINING FINISHED")
    print("=" * 70)

    print(
        f"Best Val Dice: "
        f"{best_val_dice:.6f}"
    )

    print(
        f"Test Dice at Best: "
        f"{test_dice_at_best:.6f}"
    )

## 6. One-Epoch Smoke Training

Run this optional smoke test before committing to the full training run. It checks that the complete training pipeline—not just the model forward pass—can execute successfully.

If a previous checkpoint is present and the script's resume logic detects it, remove `--resume` for a clean one-epoch run.


In [ ]:
%cd /kaggle/working/EMCAD-main

!python train_polyp.py \
    --epoch 1 \
    --batchsize 4 \
    --test_batchsize 4


## 7. Full 200-Epoch Training

The recorded reproduction used a maximum of **200 epochs** and a batch size of **4** on a Kaggle Tesla T4.

> **Note:** The experiment command below preserves the notebook's original `--resume` behavior. For a completely fresh reproduction with no existing checkpoint, omit `--resume`.


In [ ]:
%cd /kaggle/working/EMCAD-main

!python train_polyp.py \
    --epoch 200 \
    --batchsize 4 \
    --test_batchsize 4 \
    --resume


## 8. Inspect Training Artifacts and Checkpoint

The best checkpoint is selected using validation Dice. The saved full checkpoint contains the model state and training state needed for resumption.


In [ ]:
CHECKPOINT_DIR = WORK_ROOT / "checkpoints"

print("Checkpoint files:")
for path in sorted(CHECKPOINT_DIR.glob("*")):
    print(f"  {path.name}")

BEST_CHECKPOINT = CHECKPOINT_DIR / "EMCAD_ClinicDB_best_full.pth"
LAST_CHECKPOINT = CHECKPOINT_DIR / "EMCAD_ClinicDB_last_full.pth"

assert BEST_CHECKPOINT.is_file(), "Best checkpoint was not created."

print("\nBest checkpoint:", BEST_CHECKPOINT)
print("Size (MB):", BEST_CHECKPOINT.stat().st_size / (1024 ** 2))


In [ ]:
# Read the metadata stored in the best checkpoint.

checkpoint = torch.load(BEST_CHECKPOINT, map_location="cpu")

print("Checkpoint keys:")
print(sorted(checkpoint.keys()))

print("\nBest epoch:", checkpoint.get("epoch"))
print("Best validation Dice:", checkpoint.get("best_val_dice"))
print("Test Dice recorded at best checkpoint:",
      checkpoint.get("test_dice_at_best"))


## 9. Standalone Final Test Evaluation

To make the final benchmark auditable, the saved **best validation checkpoint** is evaluated separately on the ClinicDB test set.

This cell uses the evaluation script from the experiment and writes:

- per-image Dice, IoU, sensitivity, specificity, precision, and HD95;
- mean test metrics;
- prediction masks.

The standalone test evaluation is treated as the authoritative final result.


In [ ]:
%%writefile /kaggle/working/EMCAD-main/test_polyp.py

import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm

from lib.networks import EMCADNet
from utils.dataloader_polyp import get_loader
from medpy.metric.binary import hd95


# ============================================================
# EXACT KAGGLE PATHS
# ============================================================

ROOT = "/kaggle/working/EMCAD-main"

DATA_ROOT = (
    "/kaggle/input/datasets/audityghosh/"
    "emcad-polyp-pretrained/EMCAD-main/"
    "data/polyp/polyp"
)

TEST_ROOT = (
    f"{DATA_ROOT}/ClinicDB/test"
)

CHECKPOINT = (
    f"{ROOT}/checkpoints/"
    "EMCAD_ClinicDB_best_full.pth"
)

PREDICTION_ROOT = (
    f"{ROOT}/predictions_final"
)

RESULT_ROOT = (
    f"{ROOT}/results_final"
)

os.makedirs(PREDICTION_ROOT, exist_ok=True)
os.makedirs(RESULT_ROOT, exist_ok=True)


# ============================================================
# METRICS
# ============================================================

def dice_coefficient(predicted, labels):

    predicted = predicted.float()
    labels = labels.float().to(predicted.device)

    smooth = 1e-6

    predicted_flat = predicted.contiguous().view(-1)
    labels_flat = labels.contiguous().view(-1)

    intersection = (
        predicted_flat * labels_flat
    ).sum()

    total = (
        predicted_flat.sum()
        + labels_flat.sum()
    )

    return (
        2.0 * intersection + smooth
    ) / (
        total + smooth
    )


def iou(predicted, labels):

    predicted = predicted.float()
    labels = labels.float().to(predicted.device)

    smooth = 1e-6

    predicted_flat = predicted.contiguous().view(-1)
    labels_flat = labels.contiguous().view(-1)

    intersection = (
        predicted_flat * labels_flat
    ).sum()

    union = (
        predicted_flat.sum()
        + labels_flat.sum()
        - intersection
    )

    return (
        intersection + smooth
    ) / (
        union + smooth
    )


def get_binary_metrics(pred, gt):

    tp = (pred * gt).sum().item()

    tn = (
        (1 - pred) * (1 - gt)
    ).sum().item()

    fp = (
        pred * (1 - gt)
    ).sum().item()

    fn = (
        (1 - pred) * gt
    ).sum().item()

    sensitivity = (
        tp / (tp + fn + 1e-8)
    )

    specificity = (
        tn / (tn + fp + 1e-8)
    )

    precision = (
        tp / (tp + fp + 1e-8)
    )

    try:

        if pred.sum() > 0 and gt.sum() > 0:

            hd_val = hd95(
                pred.cpu().numpy(),
                gt.cpu().numpy()
            )

        else:

            hd_val = 100.0

    except Exception:

        hd_val = 100.0

    return (
        sensitivity,
        specificity,
        precision,
        hd_val
    )


# ============================================================
# LOAD MODEL
# ============================================================

print("=" * 70)
print("EMCAD FINAL TEST EVALUATION")
print("=" * 70)

print("\nCheckpoint:")
print(CHECKPOINT)

print("\nTest root:")
print(TEST_ROOT)

assert os.path.isfile(
    CHECKPOINT
), f"Checkpoint not found: {CHECKPOINT}"

assert os.path.isdir(
    TEST_ROOT
), f"Test directory not found: {TEST_ROOT}"


# ============================================================
# CREATE MODEL
# ============================================================

model = EMCADNet(
    num_classes=1,
    kernel_sizes=[1, 3, 5],
    expansion_factor=2,
    dw_parallel=True,
    add=True,
    lgag_ks=3,
    activation="relu6",
    encoder="pvt_v2_b2",
    pretrain=False
).cuda()


# ============================================================
# LOAD FULL CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cuda"
)

print("\nCheckpoint keys:")
print(checkpoint.keys())

if "model_state_dict" in checkpoint:

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )

    print("\nFull checkpoint loaded successfully.")

    print(
        "Best Epoch:",
        checkpoint["epoch"]
    )

    print(
        "Best Val Dice:",
        checkpoint["best_val_dice"]
    )

    print(
        "Test Dice recorded during training:",
        checkpoint["test_dice_at_best"]
    )

else:

    model.load_state_dict(
        checkpoint,
        strict=True
    )

    print(
        "\nPlain model checkpoint loaded."
    )


model.eval()


# ============================================================
# DATA LOADER
# ============================================================

image_root = f"{TEST_ROOT}/images/"
gt_root = f"{TEST_ROOT}/masks/"

print("\nImages:")
print(image_root)

print("\nMasks:")
print(gt_root)

assert os.path.isdir(
    image_root
)

assert os.path.isdir(
    gt_root
)


test_loader = get_loader(
    image_root=image_root,
    gt_root=gt_root,
    batchsize=1,
    trainsize=352,
    shuffle=False,
    split="test",
    color_image=True
)


print(
    "\nTest images:",
    len(test_loader.dataset)
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

prediction_dir = (
    f"{PREDICTION_ROOT}/ClinicDB_test"
)

os.makedirs(
    prediction_dir,
    exist_ok=True
)


# ============================================================
# INFERENCE
# ============================================================

DSC = 0.0
IOU = 0.0

total_images = 0

detailed_results = []


with torch.no_grad():

    for pack in tqdm(
        test_loader,
        desc="Final ClinicDB Test"
    ):

        images, gts, original_shapes, names = pack

        images = images.cuda(
            non_blocking=True
        )

        gts = gts.float().cuda(
            non_blocking=True
        )

        outputs = model(images)

        if not isinstance(outputs, list):

            outputs = [outputs]

        predictions = outputs[-1]


        for i in range(
            len(images)
        ):

            h_orig = int(
                original_shapes[0][i]
            )

            w_orig = int(
                original_shapes[1][i]
            )

            # ------------------------------------------------
            # Prediction
            # ------------------------------------------------

            p = predictions[i].unsqueeze(0)

            pred_resized = F.interpolate(
                p,
                size=(
                    h_orig,
                    w_orig
                ),
                mode="bilinear",
                align_corners=False
            )

            pred_resized = torch.sigmoid(
                pred_resized
            ).squeeze()


            # Same normalization used in training evaluation
            pred_resized = (
                (
                    pred_resized
                    - pred_resized.min()
                )
                /
                (
                    pred_resized.max()
                    - pred_resized.min()
                    + 1e-8
                )
            )


            # ------------------------------------------------
            # Ground truth
            # ------------------------------------------------

            g = gts[i].unsqueeze(0)

            gt_resized = F.interpolate(
                g,
                size=(
                    h_orig,
                    w_orig
                ),
                mode="nearest"
            ).squeeze()


            # ------------------------------------------------
            # Binary masks
            # ------------------------------------------------

            input_binary = (
                pred_resized >= 0.5
            ).float()

            target_binary = (
                gt_resized >= 0.2
            ).float()


            # ------------------------------------------------
            # Metrics
            # ------------------------------------------------

            d = dice_coefficient(
                input_binary,
                target_binary
            ).item()

            io = iou(
                input_binary,
                target_binary
            ).item()

            sens, spec, prec, hd = (
                get_binary_metrics(
                    input_binary,
                    target_binary
                )
            )


            DSC += d
            IOU += io

            total_images += 1


            # ------------------------------------------------
            # Name
            # ------------------------------------------------

            name = names[i]

            detailed_results.append({

                "Name": name,

                "Dice": d,

                "IoU": io,

                "Sensitivity": sens,

                "Specificity": spec,

                "Precision": prec,

                "HD95": hd

            })


            # ------------------------------------------------
            # Save prediction
            # ------------------------------------------------

            pred_img = (
                input_binary
                .cpu()
                .numpy()
                * 255
            ).astype(np.uint8)

            cv2.imwrite(
                os.path.join(
                    prediction_dir,
                    name
                ),
                pred_img
            )


# ============================================================
# FINAL RESULTS
# ============================================================

mean_dice = (
    DSC / total_images
)

mean_iou = (
    IOU / total_images
)

df = pd.DataFrame(
    detailed_results
)


mean_row = (
    df.mean(
        numeric_only=True
    ).to_dict()
)

mean_row["Name"] = "AVERAGE"


df = pd.concat(
    [
        df,
        pd.DataFrame([mean_row])
    ],
    ignore_index=True
)


# ============================================================
# SAVE EXCEL
# ============================================================

excel_path = (
    f"{RESULT_ROOT}/"
    "EMCAD_ClinicDB_FINAL_TEST.xlsx"
)

df.to_excel(
    excel_path,
    index=False
)


# ============================================================
# SAVE SUMMARY
# ============================================================

summary = {

    "Best Epoch":
        checkpoint.get(
            "epoch",
            "N/A"
        ),

    "Best Validation Dice":
        checkpoint.get(
            "best_val_dice",
            "N/A"
        ),

    "Training Test Dice":
        checkpoint.get(
            "test_dice_at_best",
            "N/A"
        ),

    "Standalone Test Dice":
        mean_dice,

    "Standalone Test IoU":
        mean_iou,

    "Sensitivity":
        mean_row["Sensitivity"],

    "Specificity":
        mean_row["Specificity"],

    "Precision":
        mean_row["Precision"],

    "HD95":
        mean_row["HD95"],

    "Number of Test Images":
        total_images
}


summary_df = pd.DataFrame(
    [summary]
)


summary_path = (
    f"{RESULT_ROOT}/"
    "EMCAD_ClinicDB_FINAL_SUMMARY.xlsx"
)

summary_df.to_excel(
    summary_path,
    index=False
)


# ============================================================
# PRINT FINAL RESULT
# ============================================================

print("\n")
print("=" * 70)
print("FINAL TEST RESULT")
print("=" * 70)

print(
    f"Best Epoch           : "
    f"{checkpoint.get('epoch', 'N/A')}"
)

print(
    f"Best Validation Dice : "
    f"{checkpoint.get('best_val_dice', 'N/A')}"
)

print(
    f"Training Test Dice   : "
    f"{checkpoint.get('test_dice_at_best', 'N/A')}"
)

print(
    f"Standalone Test Dice : "
    f"{mean_dice:.6f}"
)

print(
    f"Standalone Test IoU  : "
    f"{mean_iou:.6f}"
)

print(
    f"Sensitivity          : "
    f"{mean_row['Sensitivity']:.6f}"
)

print(
    f"Specificity          : "
    f"{mean_row['Specificity']:.6f}"
)

print(
    f"Precision            : "
    f"{mean_row['Precision']:.6f}"
)

print(
    f"HD95                 : "
    f"{mean_row['HD95']:.6f}"
)

print(
    f"Test Images          : "
    f"{total_images}"
)

print("\nExcel:")
print(excel_path)

print("\nPredictions:")
print(prediction_dir)

print("\nSummary:")
print(summary_path)

print("\n" + "=" * 70)
print("FINAL EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
%cd /kaggle/working/EMCAD-main

!python test_polyp.py


## 10. Final Reproduction Metrics

The expected final standalone evaluation is approximately:

- **Dice:** 95.02%
- **IoU:** 90.72%
- **Sensitivity:** 95.48%
- **Specificity:** 99.69%
- **Precision:** 94.87%
- **HD95:** 8.108
- **Test images:** 62

The exact values below are read from the generated summary file rather than manually entered.


In [ ]:
summary_path = RESULT_ROOT / "EMCAD_ClinicDB_FINAL_SUMMARY.xlsx"

if summary_path.is_file():
    import pandas as pd
    summary_df = pd.read_excel(summary_path)
    display(summary_df)
else:
    print("Summary file not found yet:", summary_path)


## 11. Qualitative Analysis

The following section ranks the test images by Dice and saves representative qualitative examples covering poor, intermediate, and strong predictions.

The generated figures contain:

1. Original image
2. Ground-truth mask
3. EMCAD prediction
4. Prediction overlay


In [ ]:
# ============================================================
# EMCAD — ClinicDB Qualitative Visualization
# ============================================================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ROOT = "/kaggle/working/EMCAD-main"

DATA_ROOT = (
    "/kaggle/input/datasets/audityghosh/"
    "emcad-polyp-pretrained/EMCAD-main/"
    "data/polyp/polyp"
)

IMAGE_DIR = f"{DATA_ROOT}/ClinicDB/test/images"
MASK_DIR = f"{DATA_ROOT}/ClinicDB/test/masks"

PRED_DIR = f"{ROOT}/predictions_final/ClinicDB_test"

OUT_DIR = f"{ROOT}/qualitative_results"
os.makedirs(OUT_DIR, exist_ok=True)

print("Images :", IMAGE_DIR)
print("Masks  :", MASK_DIR)
print("Pred   :", PRED_DIR)
print("Output :", OUT_DIR)

# ------------------------------------------------------------
# Find common images
# ------------------------------------------------------------

image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
])

print("\nTotal test images:", len(image_files))

# ------------------------------------------------------------
# Calculate Dice for selecting examples
# ------------------------------------------------------------

def calculate_dice(pred, gt):
    pred = pred > 127
    gt = gt > 127

    intersection = np.logical_and(pred, gt).sum()

    return (
        2.0 * intersection
        / (pred.sum() + gt.sum() + 1e-8)
    )


results = []

for name in image_files:

    image_path = os.path.join(IMAGE_DIR, name)
    mask_path = os.path.join(MASK_DIR, name)
    pred_path = os.path.join(PRED_DIR, name)

    if not os.path.exists(mask_path):
        continue

    if not os.path.exists(pred_path):
        continue

    gt = cv2.imread(
        mask_path,
        cv2.IMREAD_GRAYSCALE
    )

    pred = cv2.imread(
        pred_path,
        cv2.IMREAD_GRAYSCALE
    )

    if gt is None or pred is None:
        continue

    # Ensure same size
    if pred.shape != gt.shape:
        pred = cv2.resize(
            pred,
            (gt.shape[1], gt.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    dice = calculate_dice(pred, gt)

    results.append({
        "name": name,
        "dice": dice
    })


# ------------------------------------------------------------
# Sort by Dice
# ------------------------------------------------------------

results = sorted(
    results,
    key=lambda x: x["dice"]
)

print("Valid predictions:", len(results))

print("\nWorst 5:")
for r in results[:5]:
    print(f"{r['name']:30s} Dice = {r['dice']:.4f}")

print("\nBest 5:")
for r in results[-5:][::-1]:
    print(f"{r['name']:30s} Dice = {r['dice']:.4f}")


# ============================================================
# Select representative examples
# ============================================================

# 2 worst
# 2 middle
# 2 best

n = len(results)

selected = []

if n >= 6:

    selected = (
        results[:2]
        + results[n//2-1:n//2+1]
        + results[-2:]
    )

else:

    selected = results


print("\nSelected examples:")

for r in selected:
    print(
        f"{r['name']} -> Dice {r['dice']:.4f}"
    )


# ============================================================
# Create visualizations
# ============================================================

for idx, r in enumerate(selected):

    name = r["name"]

    image_path = os.path.join(
        IMAGE_DIR,
        name
    )

    mask_path = os.path.join(
        MASK_DIR,
        name
    )

    pred_path = os.path.join(
        PRED_DIR,
        name
    )

    image = cv2.imread(
        image_path
    )

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    gt = cv2.imread(
        mask_path,
        cv2.IMREAD_GRAYSCALE
    )

    pred = cv2.imread(
        pred_path,
        cv2.IMREAD_GRAYSCALE
    )

    if pred.shape != gt.shape:

        pred = cv2.resize(
            pred,
            (gt.shape[1], gt.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    # --------------------------------------------------------
    # Overlay prediction on original
    # --------------------------------------------------------

    pred_binary = pred > 127

    overlay = image.copy()

    # mark predicted region
    overlay[pred_binary] = (
        0.6 * overlay[pred_binary]
        + 0.4 * np.array([255, 0, 0])
    ).astype(np.uint8)

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    fig = plt.figure(
        figsize=(16, 4)
    )

    ax1 = plt.subplot(1, 4, 1)
    ax1.imshow(image)
    ax1.set_title("Original Image")
    ax1.axis("off")

    ax2 = plt.subplot(1, 4, 2)
    ax2.imshow(gt, cmap="gray")
    ax2.set_title("Ground Truth")
    ax2.axis("off")

    ax3 = plt.subplot(1, 4, 3)
    ax3.imshow(pred, cmap="gray")
    ax3.set_title(
        f"EMCAD Prediction\nDice = {r['dice']:.4f}"
    )
    ax3.axis("off")

    ax4 = plt.subplot(1, 4, 4)
    ax4.imshow(overlay)
    ax4.set_title("Prediction Overlay")
    ax4.axis("off")

    plt.suptitle(
        name,
        fontsize=12
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    save_name = (
        f"{idx+1:02d}_{os.path.splitext(name)[0]}.png"
    )

    save_path = os.path.join(
        OUT_DIR,
        save_name
    )

    plt.savefig(
        save_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    plt.close(fig)


print("\n" + "=" * 60)
print("QUALITATIVE VISUALIZATION COMPLETE")
print("=" * 60)

print(
    "Saved to:",
    OUT_DIR
)

## 12. Reproduction Summary

### Published benchmark vs. reproduction

| Metric / setting | Published EMCAD | This reproduction |
|---|---:|---:|
| Model | PVT-EMCAD-B2 | PVT-EMCAD-B2 |
| Dataset | ClinicDB | ClinicDB |
| Input | 352 × 352 | 352 × 352 |
| Training | 200 epochs | 200 epochs |
| Batch size | 16 | 4 |
| GPU | RTX A6000 48 GB | Tesla T4 |
| Dice | **95.21%** | **95.02%** |
| Runs | 5-run average | 1 run |

### Interpretation

The reproduction differs from the published Dice by approximately **0.19 percentage points**. The result is therefore closely aligned with the reported benchmark, while the reduced batch size and different GPU mean that the experiment should be described as an **implementation-based reproduction**, not an exact reproduction.

### Key reproducibility artifacts

- Training notebook
- Adapted training script
- Adapted standalone test script
- Best full checkpoint
- Per-image test metrics
- Final summary metrics
- Qualitative prediction figures

## 13. Attribution

This repository uses the authors' released EMCAD implementation. The original work and authors should be cited when using this code or reproduction.

**Original paper:**  
Rahman, M. M., Munir, M., & Marculescu, R. *EMCAD: Efficient Multi-scale Convolutional Attention Decoding for Medical Image Segmentation.* CVPR 2024.
